<img src="https://images.squarespace-cdn.com/content/v1/67351b6c456f5a62d1c1bca2/17a93d03-0cae-49bf-b152-8dfd0cdb9d7d/Quantum+Rings+Logo+Main+White.png?format=300w" alt="Quantum Rings" width="150" align="right"/>

# Optimization with QAOA
The Quantum Approximate Optimization Algorithm (QAOA) is a hybrid quantum-classical algorithm designed to solve combinatorial optimization problems. It’s particularly effective for finding approximate solutions to problems that involve finding the best configuration out of many possibilities, such as the Max-Cut problem or the Traveling Salesman Problem. QAOA leverages the principles of quantum mechanics, like superposition and entanglement, to explore multiple solutions simultaneously, while classical optimization techniques fine-tune the results. Its strength lies in potentially offering better approximations than classical algorithms in a faster time frame, showcasing quantum computing's potential for real-world optimization tasks.



Learn QAOA through the complete documentation here:
* [Solving Optimization Problems](https://www.quantumrings.com/doc/Introopt.html)
* [Quadratic Unconstrained Binary Optimizations](https://www.quantumrings.com/doc/qubo.html)
* [Quantum Approximation Optimization Algorithm (QAOA)](https://www.quantumrings.com/doc/qaoa.html)
* [Solving the Max-Cut problem with QAOA](https://www.quantumrings.com/doc/maxcut.html) <-- This is what we are doing in this notebook.

Here's a sample implementation of QAOA to solve the Max-Cut problem. The Max-Cut problem is a classic combinatorial optimization problem where, given a graph, the objective is to divide the graph's nodes into two sets such that the number of edges between the sets is maximized.

## Required One-time Setup

If you haven't already done so, you will need to obtain a free API token from the [Quantum Rings portal](https://portal.quantumrings.com/). Replace the placeholder token and email address in the provider setup cell below with your own credentials.

In [ ]:
%%capture
%pip install QuantumRingsLib matplotlib numpy scikit-optimize

In [ ]:
import QuantumRingsLib
from QuantumRingsLib import QuantumRegister, AncillaRegister, ClassicalRegister, QuantumCircuit
from QuantumRingsLib import QuantumRingsProvider
from QuantumRingsLib import job_monitor
from QuantumRingsLib import JobStatus
from matplotlib import pyplot as plt
from skopt import gp_minimize
import numpy as np
import math

In [ ]:
provider = QuantumRingsProvider(token="YOUR_TOKEN_HERE", name="YOUR_EMAIL_ADDRESS")
backend = provider.get_backend("scarlet_quantum_rings")
shots = 100

provider.active_account()

In [ ]:
# define the operator U(B, beta)
def Operator_UB(graph, beta,qc, q, n_qubits):
    for i in range(n_qubits): qc.rx(2*beta, q[i])

# define the operator U(C,gamma)
def Operator_UC(graph, gamma, qc, q, n_qubits):
    for edge in graph:
        qc.cx(q[edge[0]], q[edge[1]])
        # multiply the gamma by the weight of the edge
        qc.rz(gamma*edge[2], q[edge[1]])
        qc.cx(q[edge[0]], q[edge[1]])

# a helper routine that computes the total weight of the cuts
def WeightOfCuts(bitstring):
    totalWeight = 0
    for edge in graph:
        if bitstring[edge[0]] != bitstring[edge[1]]:
            totalWeight += edge[2]
    return totalWeight

def jobCallback(job_id, state, job):
    #print("Job Status: ", state)
    pass

# Builds the QAOA state.
def qaoaState( x, graph, p, n_qubits, expectationValue = True, shots=1024):
    gammas = []
    betas = []
    # setup the gamma and beta array
    for i in range(len(x)//2):
        gammas.append(x[i*2])
        betas.append(x[(i*2)+1])
    # Create the quantum circuit
    q = QuantumRegister(n_qubits)
    c = ClassicalRegister(n_qubits)
    qc = QuantumCircuit(q, c)

    # First set the qubits in an equal superposition state
    for i in range(n_qubits):
        qc.h(q[i])

    # Apply the gamma and beta operators in repetition
    for i in range(p):
        # Apply U(C,gamma)
        Operator_UC(graph, gammas[i], qc, q, n_qubits)

        # Apply U(B, beta)
        Operator_UB(graph, betas[i],qc, q, n_qubits)

    # Measure the qubits
    for i in range(n_qubits):
        qc.measure(q[i], c[i])

    # Execute the circuit now
    job = backend.run(qc, shots=shots)
    job.wait_for_final_state(0, 5, jobCallback)
    counts = job.result().get_counts()

    # decide what to return
    if ( True == expectationValue ):
        # Calculate the expectation value of the measurement.
        expectation = 0
        for bitstring in counts:
            probability = float(counts[bitstring]) / shots
            expectation += WeightOfCuts(bitstring) * probability
        del job
        return( expectation )
    else:
        # Just return the counts
        del job
        return(counts)

## Helper Routines

In [ ]:
def plot_histogram (counts, title=""):
    """
    Plots the histogram of the counts

    Args:

        counts (dict):
            The dictionary containing the counts of states

        titles (str):
            A title for the graph.

    Returns:
        None

    """
    fig, ax = plt.subplots(figsize =(10, 7))
    plt.xlabel("States")
    plt.ylabel("Counts")
    mylist = [key for key, val in counts.items() for _ in range(val)]

    unique, inverse = np.unique(mylist, return_inverse=True)
    bin_counts = np.bincount(inverse)

    plt.bar(unique, bin_counts)

    maxFreq = max(counts.values())
    plt.ylim(ymax=np.ceil(maxFreq / 10) * 10 if maxFreq % 10 else maxFreq + 10)
    # Show plot
    plt.title(title)
    plt.show()
    return

### Put everything together and executing the code

Note: It will take a while to execute this code.

In [ ]:
p = 2                # Number of circuit layers
n_calls = 100        # Optimization cycles
n_random_starts = 2  # See scikit documentation
dimensions = []  # Built below based on circuit layers

# Example 2-regular graphs with equal weights.
# Each graph is a list of edges and their weights.

graph = [(0, 1, 1.0), (0, 3, 1.0), (1, 2, 1.0), (2, 3, 1.0)]

# Find the number of qubits required.
n_qubits = 0
for i in range (len(graph)):
    for j in range (2):
        n_qubits = max(graph[i][j], n_qubits)
n_qubits += 1

# Construct the search space depending upon the circuit layers
for i in range(p):
    dimensions.append((0,2*np.pi))
    dimensions.append((0,np.pi))
d = tuple(dimensions)

# setup the optimization function, as its negative
f = lambda x: -1*qaoaState(x, graph, p, n_qubits)

# Use the Bayesian optimization using Gaussian Processes from Scikit optimizer
# to maximize the cost function (by minimizing its negative)
res = gp_minimize(func=f,
        dimensions = d,
        n_calls=n_calls,
        n_random_starts=n_random_starts)

# Fetch the optimal gamma and beta values
x = res.x
# Execute the qaoa state with the optimal gamma and beta values
counts = qaoaState(x, graph, p, n_qubits, False)
plot_histogram(counts, title='QAOA Max-Cut')